# 📊 Normalidade em Séries Temporais

Este notebook mostra como:
- Testar normalidade em dados independentes
- Testar normalidade em séries temporais (resíduos de modelos)
- Aplicar transformações (Box-Cox, Yeo-Johnson)
- Visualizar resultados com histogramas e QQ-plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.stattools import jarque_bera
from statsmodels.graphics.gofplots import qqplot
from sklearn.preprocessing import PowerTransformer, StandardScaler, MinMaxScaler, RobustScaler, QuantileTransformer

## 1. Testes de normalidade em dados independentes

In [ ]:
np.random.seed(42)
dados = np.random.normal(loc=0, scale=1, size=100)

# Shapiro-Wilk
stat, p = stats.shapiro(dados)
print("Shapiro-Wilk: stat=%.3f, p=%.3f" % (stat, p))

# D’Agostino & Pearson
stat, p = stats.normaltest(dados)
print("D’Agostino & Pearson: stat=%.3f, p=%.3f" % (stat, p))

# Anderson-Darling
resultado = stats.anderson(dados)
print("Anderson-Darling: stat=%.3f" % resultado.statistic)

## 2. Séries temporais: normalidade dos resíduos

In [ ]:
# Série temporal real: passageiros aéreos
dados = sm.datasets.airpassengers.load_pandas().data
serie = dados['value']

plt.figure(figsize=(10,4))
plt.plot(serie)
plt.title("Série original - Passageiros Aéreos")
plt.show()

# Ajustar modelo ARIMA
modelo = sm.tsa.ARIMA(serie, order=(1,1,1)).fit()
residuos = modelo.resid

# Testes de normalidade nos resíduos
jb_stat, jb_pvalue, _, _ = jarque_bera(residuos)
print("Jarque-Bera: stat=%.3f, p=%.3f" % (jb_stat, jb_pvalue))

stat, p = stats.shapiro(residuos)
print("Shapiro-Wilk: stat=%.3f, p=%.3f" % (stat, p))

# QQ-plot dos resíduos
qqplot(residuos, line='45', fit=True)
plt.title("QQ-plot dos resíduos")
plt.show()

## 3. Transformações: Box-Cox e Yeo-Johnson

In [ ]:
from scipy.stats import boxcox

# Box-Cox (dados > 0)
serie_boxcox, lambda_bc = boxcox(serie)
print("Lambda Box-Cox:", lambda_bc)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.hist(serie, bins=20)
plt.title("Histograma Original")

plt.subplot(1,2,2)
plt.hist(serie_boxcox, bins=20)
plt.title("Histograma Box-Cox")
plt.show()

# Yeo-Johnson (aceita negativos)
pt = PowerTransformer(method='yeo-johnson')
serie_yj = pt.fit_transform(serie.values.reshape(-1,1))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
qqplot(serie_boxcox, line='45', fit=True, ax=plt.gca())
plt.title("QQ-plot Box-Cox")

plt.subplot(1,2,2)
qqplot(serie_yj.flatten(), line='45', fit=True, ax=plt.gca())
plt.title("QQ-plot Yeo-Johnson")
plt.show()

## 4. Normalização (quando já é normal)
- Z-score (StandardScaler)
- Min-Max
- RobustScaler
- QuantileTransformer (para forçar normalidade)

In [ ]:
scalers = {
    'Z-score': StandardScaler(),
    'Min-Max': MinMaxScaler(),
    'Robust': RobustScaler(),
    'Quantile->Normal': QuantileTransformer(output_distribution='normal')
}

for nome, scaler in scalers.items():
    transformado = scaler.fit_transform(serie.values.reshape(-1,1))
    print(f"{nome}: média={np.mean(transformado):.2f}, desvio={np.std(transformado):.2f}")